# rank-world-size-args — faded example 2: Reduce: dst receives from all non-dst ranks

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `rank-world-size-args`. The last cell reports your progress on the `Distributed: rank/world_size args` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: rank/world_size args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rank-world-size-args`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rank-world-size-args"
DD_SUBTOPIC = "Distributed: rank/world_size args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In a reduce operation, every non-destination rank sends its tensor to `dst`, and `dst` receives from all other ranks. This is the mirror image of broadcast: one destination, all senders. The signature `reduce_protocol(tensor, rank, world_size, dst=0)` follows the same convention as broadcast, with `dst` instead of `src`.

## Faded exercise 2

### Exercise — Reduce protocol: dst receives from all non-dst ranks

Complete `reduce_protocol(tensor, rank, world_size, dst=0)`. When `rank == dst`, return a list of receive actions from every other rank. When `rank != dst`, return a single send to `dst`.

Fill in the dst-rank receive list.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

def reduce_protocol(tensor, rank: int, world_size: int, dst: int = 0) -> list:
    if rank == dst:
        return [('recv', other) for other in range(world_size) if other != dst]
    return [('send', dst)]

for r in range(3):
    print(r, reduce_protocol(None, r, 3))


def _test():
    # Dst=0, world_size=4
    dst_actions = reduce_protocol(None, 0, 4)
    assert dst_actions == [('recv', 1), ('recv', 2), ('recv', 3)]
    # Non-dst ranks
    for r in range(1, 4):
        actions = reduce_protocol(None, r, 4)
        assert actions == [('send', 0)], f'rank {r} should send to 0'
    # Non-default dst
    dst2_actions = reduce_protocol(None, 2, 4, dst=2)
    assert ('recv', 0) in dst2_actions
    assert ('recv', 1) in dst2_actions
    assert ('recv', 3) in dst2_actions
    assert len(dst2_actions) == 3


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def reduce_protocol(tensor, rank: int, world_size: int, dst: int = 0) -> list:
    if rank == dst:
        return [('recv', other) for other in range(world_size) if other != dst]
    return [('send', dst)]

for r in range(3):
    print(r, reduce_protocol(None, r, 3))
```
</details>